# 08 — Siamese CNN V3 with FCGR k=7

## Obiettivo

Valutare l'effetto dell'aumento della dimensione dei k-mer nella
rappresentazione FCGR mantenendo invariata la pipeline Siamese.

L'esperimento precedente con k=6 ha ottenuto:

- Pairwise ROC-AUC ≈ 0.665
- Pairwise same/different accuracy ≈ 62%
- 12-way prototype accuracy ≈ 28.9%
- 12-way Top-20 reference-bank accuracy ≈ 30.8%
- 12-way Top-20 Macro-F1 ≈ 28.7%

In questo notebook viene utilizzato:

- FCGR k=7
- dimensione FCGR: 128 × 128
- 12 classi Tumor + Healthy
- CNN V3 condivisa
- embedding L2-normalizzato da 128 dimensioni
- Euclidean distance
- Contrastive Loss
- 50% positive pairs
- 50% random negative pairs
- downsampling del training set a 2111 campioni/classe

L'obiettivo iniziale è isolare l'effetto di k=7 rispetto alla baseline
k=6, senza introdurre contemporaneamente altre modifiche al training.

In [1]:
# ============================================================
# IMPORTS + CONFIG
# ============================================================

from pathlib import Path

import random
import time
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve
)


# ============================================================
# PROJECT ROOT
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_k7_tumor_healthy"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# DATA PATHS
# ============================================================

MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


# ============================================================
# FCGR k=7
# ============================================================

K = 7

FCGR_SIZE = 2 ** K


FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


# ============================================================
# EXPERIMENT
# ============================================================

RANDOM_STATE = 42

N_CLASSES = 12

EMBEDDING_DIM = 128

EUCLIDEAN_MARGIN = 1.25


# Stessi numeri della baseline k=6
TRAIN_PAIRS_PER_EPOCH = 50_000

VAL_PAIRS = 10_000

POSITIVE_FRACTION = 0.50


# k=7 richiede più memoria di k=6
BATCH_SIZE = 64


LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


AMP_ENABLED = (
    DEVICE.type == "cuda"
)


print("=" * 72)
print("EXPERIMENT CONFIGURATION")
print("=" * 72)

print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print()
print("FCGR k:", K)
print("Expected FCGR size:", FCGR_SIZE, "x", FCGR_SIZE)
print("Embedding dim:", EMBEDDING_DIM)
print("Batch size:", BATCH_SIZE)
print("Margin:", EUCLIDEAN_MARGIN)

EXPERIMENT CONFIGURATION
Project: D:\Daria\Desktop\eccdna_fcgr_siamese
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

FCGR k: 7
Expected FCGR size: 128 x 128
Embedding dim: 128
Batch size: 64
Margin: 1.25


In [2]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(
    RANDOM_STATE
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print(
    "Random seed:",
    RANDOM_STATE
)

Random seed: 42


In [3]:
# ============================================================
# LOAD 12-WAY MANIFEST
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


print("=" * 72)
print("12-WAY TUMOR + HEALTHY DATASET")
print("=" * 72)

print(
    "Samples:",
    len(metadata)
)

print(
    "Classes:",
    metadata["class_id"].nunique()
)

print()

display(
    class_mapping
)

12-WAY TUMOR + HEALTHY DATASET
Samples: 625937
Classes: 12



,original_class_id,disease_clean,disease_group,class_id
0,0,gastric cancer,cancer,0
1,1,healthy,healthy,1
2,2,ovarian cancer,cancer,2
3,3,prostate cancer,cancer,3
4,4,colorectal cancer,cancer,4
5,5,lymphoma,cancer,5
6,7,cervical adenocarcinoma,cancer,6
7,8,leukemia,cancer,7
8,11,hypopharyngeal squamous cell carcinoma,cancer,8
9,12,glioblastoma cancer,cancer,9


In [4]:
# ============================================================
# TRAIN SPLIT
# ============================================================

full_train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


train_counts = (
    full_train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
)


print("=" * 72)
print("ORIGINAL TRAIN DISTRIBUTION")
print("=" * 72)

display(
    train_counts.rename(
        "samples"
    ).to_frame()
)


# ============================================================
# DOWNSAMPLING
# ============================================================

MIN_CLASS_SIZE = int(
    train_counts.min()
)


balanced_parts = []


for class_id in sorted(
    full_train_metadata[
        "class_id"
    ].unique()
):

    class_df = (
        full_train_metadata[
            full_train_metadata[
                "class_id"
            ]
            ==
            class_id
        ]
    )


    sampled = class_df.sample(
        n=MIN_CLASS_SIZE,
        replace=False,
        random_state=(
            RANDOM_STATE
            +
            int(class_id)
        )
    )


    balanced_parts.append(
        sampled
    )


train_metadata = (
    pd.concat(
        balanced_parts,
        ignore_index=True
    )
    .reset_index(drop=True)
)


balanced_counts = (
    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
)


print()
print("=" * 72)
print("BALANCED TRAIN")
print("=" * 72)

print(
    "Full train:",
    len(full_train_metadata)
)

print(
    "Samples/class:",
    MIN_CLASS_SIZE
)

print(
    "Balanced train:",
    len(train_metadata)
)

print(
    "Classes:",
    train_metadata[
        "class_id"
    ].nunique()
)

print()

display(
    balanced_counts.rename(
        "samples"
    ).to_frame()
)

ORIGINAL TRAIN DISTRIBUTION


,samples
class_id,
0,10000
1,10000
2,10000
3,10000
4,10000
5,10000
6,10000
7,10000
8,5052



BALANCED TRAIN
Full train: 96167
Samples/class: 2111
Balanced train: 25332
Classes: 12



,samples
class_id,
0,2111
1,2111
2,2111
3,2111
4,2111
5,2111
6,2111
7,2111
8,2111


In [5]:
# ============================================================
# VALIDATION POOL — SAME AS PREVIOUS EXPERIMENTS
# ============================================================

old_to_new = dict(
    zip(
        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


included_original_ids = set(
    old_to_new.keys()
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original[
        "class_id"
    ]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original[
            "class_id"
        ].isin(
            included_original_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = (
    val_metadata[
        "class_id"
    ]
)


val_metadata[
    "class_id"
] = (
    val_metadata[
        "original_class_id"
    ]
    .map(
        old_to_new
    )
    .astype(int)
)


print("=" * 72)
print("VALIDATION POOL")
print("=" * 72)

print(
    "Samples:",
    len(val_metadata)
)

print(
    "Classes:",
    val_metadata[
        "class_id"
    ].nunique()
)

print()

display(
    val_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("samples")
    .to_frame()
)

VALIDATION POOL
Samples: 9753
Classes: 12



,samples
class_id,
0,1000
1,1000
2,1000
3,1000
4,1000
5,1000
6,1000
7,1000
8,585


In [6]:
# ============================================================
# LOAD FCGR k=7 CACHE
# ============================================================

print(
    "FCGR path:",
    FCGR_PATH
)

print(
    "Index path:",
    FCGR_INDEX_PATH
)

print()


assert FCGR_PATH.exists(), (
    f"FCGR k=7 non trovata: {FCGR_PATH}"
)

assert FCGR_INDEX_PATH.exists(), (
    f"Indice FCGR k=7 non trovato: {FCGR_INDEX_PATH}"
)


fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


print("=" * 72)
print("FCGR k=7 CACHE")
print("=" * 72)

print(
    "FCGR shape:",
    fcgr_memmap.shape
)

print(
    "FCGR dtype:",
    fcgr_memmap.dtype
)

print(
    "Index rows:",
    len(fcgr_index)
)


# ============================================================
# CHECK DIMENSION
# ============================================================

assert (
    fcgr_memmap.shape[1]
    ==
    FCGR_SIZE
)

assert (
    fcgr_memmap.shape[2]
    ==
    FCGR_SIZE
)


# ============================================================
# CHECK COVERAGE
# ============================================================

missing_train = (
    ~train_metadata["id"]
    .isin(
        id_to_fcgr_row
    )
).sum()


missing_val = (
    ~val_metadata["id"]
    .isin(
        id_to_fcgr_row
    )
).sum()


print()
print(
    "Missing train:",
    missing_train
)

print(
    "Missing validation:",
    missing_val
)


assert missing_train == 0

assert missing_val == 0


print()
print(
    "FCGR k=7 cache: OK"
)

FCGR path: D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\fcgr_cache\fcgr_k7.npy
Index path: D:\Daria\Desktop\eccdna_fcgr_siamese\data\processed\fcgr_cache\fcgr_k7_index.tsv

FCGR k=7 CACHE
FCGR shape: (150272, 128, 128)
FCGR dtype: float32
Index rows: 150272

Missing train: 0
Missing validation: 0

FCGR k=7 cache: OK


In [7]:
# ============================================================
# CNN ENCODER V3
# ============================================================

class FCGRCNNEncoderV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.MaxPool2d(2),


            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                128
            ),

            nn.ReLU(
                inplace=True
            ),


            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.features(
            x
        )


        z = self.embedding_head(
            x
        )


        return F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )

In [8]:
# ============================================================
# SIAMESE V3
# ============================================================

class SiameseV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.encoder = FCGRCNNEncoderV3(
            embedding_dim=embedding_dim
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = (
            x1.shape[0]
        )


        # Un solo forward grande è più efficiente
        # ma l'encoder è ESATTAMENTE lo stesso.

        x = torch.cat(
            [
                x1,
                x2
            ],
            dim=0
        )


        z = self.encoder(
            x
        )


        z1 = z[
            :batch_size
        ]

        z2 = z[
            batch_size:
        ]


        return (
            z1,
            z2
        )


model = SiameseV3(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)


n_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("=" * 72)
print("SIAMESE V3 k=7")
print("=" * 72)

print(
    "Trainable parameters:",
    f"{n_parameters:,}"
)

print(
    "Training from scratch:",
    True
)

SIAMESE V3 k=7
Trainable parameters: 807,264
Training from scratch: True


In [9]:
# ============================================================
# ENCODER k=7 SANITY CHECK
# ============================================================

sample_rows = (
    train_metadata[
        "id"
    ]
    .iloc[:8]
    .map(
        id_to_fcgr_row
    )
    .to_numpy(
        dtype=np.int64
    )
)


x = np.stack(
    [
        np.array(
            fcgr_memmap[
                int(row)
            ],
            dtype=np.float32,
            copy=True
        )
        for row in sample_rows
    ],
    axis=0
)


x = torch.from_numpy(
    x
).unsqueeze(1)


x = x.to(
    DEVICE
)


model.eval()


with torch.no_grad():

    z = model.encoder(
        x
    )


print("=" * 72)
print("k=7 ENCODER SANITY CHECK")
print("=" * 72)

print(
    "Input:",
    x.shape
)

print(
    "Embedding:",
    z.shape
)

print(
    "Mean embedding norm:",
    z.norm(
        dim=1
    ).mean().item()
)

print(
    "Finite:",
    torch.isfinite(z)
    .all()
    .item()
)

k=7 ENCODER SANITY CHECK
Input: torch.Size([8, 1, 128, 128])
Embedding: torch.Size([8, 128])
Mean embedding norm: 1.0
Finite: True


In [10]:
# ============================================================
# RANDOM SIAMESE PAIR DATASET
# ============================================================

class RandomSiamesePairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        positive_fraction=0.50,
        seed=42,
        dynamic=True
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.n_pairs = int(n_pairs)

        self.positive_fraction = float(
            positive_fraction
        )

        self.seed = int(seed)

        self.dynamic = bool(dynamic)


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


        self.classes = np.array(
            sorted(
                np.unique(self.labels)
            ),
            dtype=np.int64
        )


        self.class_to_indices = {
            int(c):
                np.where(
                    self.labels == c
                )[0]

            for c in self.classes
        }


        self._generate_pairs(
            self.seed
        )


    def _generate_pairs(
        self,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )


        n_positive = int(
            round(
                self.n_pairs
                *
                self.positive_fraction
            )
        )


        targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )

        targets[:n_positive] = 1.0

        rng.shuffle(
            targets
        )


        anchors = rng.integers(
            0,
            len(self.labels),
            size=self.n_pairs
        )


        partners = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                anchors[i]
            )

            anchor_class = int(
                self.labels[
                    anchor_idx
                ]
            )


            # =================================================
            # POSITIVE
            # =================================================

            if targets[i] == 1.0:

                candidates = (
                    self.class_to_indices[
                        anchor_class
                    ]
                )


                partner_idx = anchor_idx


                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(
                        rng.choice(
                            candidates
                        )
                    )


            # =================================================
            # NEGATIVE
            # =================================================

            else:

                negative_classes = (
                    self.classes[
                        self.classes
                        !=
                        anchor_class
                    ]
                )


                negative_class = int(
                    rng.choice(
                        negative_classes
                    )
                )


                partner_idx = int(
                    rng.choice(
                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            partners[i] = (
                partner_idx
            )


        self.row1 = self.rows[
            anchors
        ]

        self.row2 = self.rows[
            partners
        ]

        self.targets = targets


    def set_epoch(
        self,
        epoch
    ):

        if self.dynamic:

            self._generate_pairs(

                self.seed
                +
                int(epoch)
                *
                100_003
            )


    def __len__(self):

        return self.n_pairs


    def __getitem__(
        self,
        index
    ):

        x1 = np.array(
            self.fcgr_memmap[
                int(
                    self.row1[index]
                )
            ],
            dtype=np.float32,
            copy=True
        )


        x2 = np.array(
            self.fcgr_memmap[
                int(
                    self.row2[index]
                )
            ],
            dtype=np.float32,
            copy=True
        )


        return {
            "x1":
                torch.from_numpy(x1)
                .unsqueeze(0),

            "x2":
                torch.from_numpy(x2)
                .unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                )
        }

In [11]:
# ============================================================
# BUILD PAIR DATASETS
# ============================================================

train_pair_dataset = RandomSiamesePairDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=TRAIN_PAIRS_PER_EPOCH,
    positive_fraction=POSITIVE_FRACTION,
    seed=RANDOM_STATE,
    dynamic=True
)


val_pair_dataset = RandomSiamesePairDataset(
    metadata=val_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=VAL_PAIRS,
    positive_fraction=0.50,
    seed=RANDOM_STATE + 50_000,
    dynamic=False
)


train_pair_loader = DataLoader(
    train_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


val_pair_loader = DataLoader(
    val_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


print("=" * 72)
print("PAIR DATASETS")
print("=" * 72)

print(
    "Train pairs:",
    len(train_pair_dataset)
)

print(
    "Train positive fraction:",
    f"{train_pair_dataset.targets.mean():.4f}"
)

print(
    "Validation pairs:",
    len(val_pair_dataset)
)

print(
    "Validation positive fraction:",
    f"{val_pair_dataset.targets.mean():.4f}"
)

print(
    "Train batches:",
    len(train_pair_loader)
)

PAIR DATASETS
Train pairs: 50000
Train positive fraction: 0.5000
Validation pairs: 10000
Validation positive fraction: 0.5000
Train batches: 782


In [12]:
# ============================================================
# EUCLIDEAN CONTRASTIVE LOSS
# ============================================================

class EuclideanContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        z1 = z1.float()

        z2 = z2.float()

        target = target.float()


        distances = torch.linalg.vector_norm(
            z1 - z2,
            ord=2,
            dim=1
        )


        positive_loss = (
            target
            *
            distances.pow(2)
        )


        negative_loss = (
            (1.0 - target)
            *
            F.relu(
                self.margin
                -
                distances
            ).pow(2)
        )


        loss = (
            positive_loss
            +
            negative_loss
        ).mean()


        return (
            loss,
            distances
        )


criterion = EuclideanContrastiveLoss(
    margin=EUCLIDEAN_MARGIN
)


print(
    "Euclidean margin:",
    criterion.margin
)

Euclidean margin: 1.25


In [13]:
# ============================================================
# PAIRWISE EVALUATION
# ============================================================

def evaluate_pairwise(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_targets = []
    all_distances = []


    with torch.no_grad():

        for batch in loader:

            x1 = batch["x1"].to(
                DEVICE,
                non_blocking=True
            )

            x2 = batch["x2"].to(
                DEVICE,
                non_blocking=True
            )

            target = batch["target"].to(
                DEVICE,
                non_blocking=True
            )


            with torch.autocast(
                device_type=DEVICE.type,
                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),
                enabled=AMP_ENABLED
            ):

                z1, z2 = model(
                    x1,
                    x2
                )


            loss, distances = criterion(
                z1,
                z2,
                target
            )


            batch_size_now = (
                target.shape[0]
            )


            total_loss += (
                loss.item()
                *
                batch_size_now
            )

            total_samples += (
                batch_size_now
            )


            all_targets.append(
                target.cpu().numpy()
            )

            all_distances.append(
                distances.cpu().numpy()
            )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive = distances[
        targets == 1
    ]

    negative = distances[
        targets == 0
    ]


    d_pos = float(
        positive.mean()
    )

    d_neg = float(
        negative.mean()
    )

    gap = (
        d_neg
        -
        d_pos
    )


    pooled_variance = (
        0.5
        *
        (
            positive.var()
            +
            negative.var()
        )
    )


    d_prime = float(
        gap
        /
        np.sqrt(
            pooled_variance
            +
            1e-12
        )
    )


    auc = float(
        roc_auc_score(
            targets,
            -distances
        )
    )


    return {
        "loss":
            total_loss
            /
            total_samples,

        "auc":
            auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime
    }

In [14]:
# ============================================================
# TRAIN ONE RANDOM-PAIR EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    dataset,
    criterion,
    optimizer,
    scaler,
    epoch
):

    model.train()

    dataset.set_epoch(
        epoch
    )


    total_loss = 0.0
    total_samples = 0

    all_targets = []
    all_distances = []


    start_time = (
        time.perf_counter()
    )


    for batch in loader:

        x1 = batch["x1"].to(
            DEVICE,
            non_blocking=True
        )

        x2 = batch["x2"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),
            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        loss, distances = criterion(
            z1,
            z2,
            target
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        batch_size_now = (
            target.shape[0]
        )


        total_loss += (
            loss.detach().item()
            *
            batch_size_now
        )

        total_samples += (
            batch_size_now
        )


        all_targets.append(
            target.detach()
            .cpu()
            .numpy()
        )

        all_distances.append(
            distances.detach()
            .cpu()
            .numpy()
        )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive = distances[
        targets == 1
    ]

    negative = distances[
        targets == 0
    ]


    return {
        "loss":
            total_loss
            /
            total_samples,

        "auc":
            float(
                roc_auc_score(
                    targets,
                    -distances
                )
            ),

        "d_pos":
            float(
                positive.mean()
            ),

        "d_neg":
            float(
                negative.mean()
            ),

        "gap":
            float(
                negative.mean()
                -
                positive.mean()
            ),

        "seconds":
            (
                time.perf_counter()
                -
                start_time
            )
    }

In [15]:
# ============================================================
# k=7 GPU BENCHMARK
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


batch = next(
    iter(train_pair_loader)
)


x1 = batch["x1"].to(
    DEVICE
)

x2 = batch["x2"].to(
    DEVICE
)

target = batch["target"].to(
    DEVICE
)


if DEVICE.type == "cuda":

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


# Warm-up GPU
for _ in range(3):

    optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = model(
            x1,
            x2
        )


    loss, _ = criterion(
        z1,
        z2,
        target
    )


    scaler.scale(
        loss
    ).backward()

    scaler.step(
        optimizer
    )

    scaler.update()


if DEVICE.type == "cuda":
    torch.cuda.synchronize()


start = time.perf_counter()


N_BENCHMARK_STEPS = 10


for _ in range(
    N_BENCHMARK_STEPS
):

    optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = model(
            x1,
            x2
        )


    loss, _ = criterion(
        z1,
        z2,
        target
    )


    scaler.scale(
        loss
    ).backward()

    scaler.step(
        optimizer
    )

    scaler.update()


if DEVICE.type == "cuda":
    torch.cuda.synchronize()


elapsed = (
    time.perf_counter()
    -
    start
)


seconds_per_batch = (
    elapsed
    /
    N_BENCHMARK_STEPS
)


estimated_epoch_seconds = (
    seconds_per_batch
    *
    len(train_pair_loader)
)


if DEVICE.type == "cuda":

    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        /
        (1024 ** 3)
    )

else:

    peak_memory_gb = float("nan")


print("=" * 72)
print("k=7 TRAINING BENCHMARK")
print("=" * 72)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Batches/epoch:",
    len(train_pair_loader)
)

print(
    "Seconds/batch:",
    f"{seconds_per_batch:.3f}"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds:.1f}s"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds / 60:.2f} min"
)

print(
    "Peak GPU memory:",
    f"{peak_memory_gb:.2f} GB"
)

k=7 TRAINING BENCHMARK
Batch size: 64
Batches/epoch: 782
Seconds/batch: 0.076
Estimated epoch: 59.4s
Estimated epoch: 0.99 min
Peak GPU memory: 2.17 GB


In [16]:
# ============================================================
# RESET MODEL AFTER BENCHMARK
# ============================================================

set_seed(
    RANDOM_STATE
)


model = SiameseV3(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


print(
    "Model reset after benchmark: OK"
)

Model reset after benchmark: OK


In [17]:
# ============================================================
# k=7 RANDOM-PAIR SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


smoke_initial_state = copy.deepcopy(
    model.state_dict()
)


print("=" * 112)
print("SIAMESE V3 k=7 — RANDOM PAIRS — SMOKE TEST")
print("=" * 112)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_one_epoch(
        model=model,
        loader=train_pair_loader,
        dataset=train_pair_dataset,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch
    )


    val_metrics = evaluate_pairwise(
        model=model,
        loader=val_pair_loader,
        criterion=criterion
    )


    print(
        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

SIAMESE V3 k=7 — RANDOM PAIRS — SMOKE TEST
Epoch 01/3 | train loss 0.3799 | train AUC 0.6219 | val loss 0.3815 | val AUC 0.6269 | d+ 0.6164 | d- 0.7210 | gap 0.1046 | d' 0.4603 | 76.4s
Epoch 02/3 | train loss 0.3690 | train AUC 0.6396 | val loss 0.3778 | val AUC 0.6428 | d+ 0.6276 | d- 0.7496 | gap 0.1220 | d' 0.5171 | 69.0s
Epoch 03/3 | train loss 0.3664 | train AUC 0.6466 | val loss 0.3680 | val AUC 0.6438 | d+ 0.5710 | d- 0.6708 | gap 0.0998 | d' 0.5169 | 152.9s


In [18]:
# ============================================================
# FULL k=7 RANDOM-PAIR TRAINING
# ============================================================

MAX_EPOCHS = 40

EARLY_STOPPING_PATIENCE = 10

MIN_DELTA = 1e-4


CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "siamese_v3_k7_euclidean_best.pt"
)


HISTORY_PATH = (
    ARTIFACTS_DIR
    / "siamese_v3_k7_euclidean_history.tsv"
)


# ============================================================
# RESTORE EXACT INITIAL MODEL
# ============================================================

model.load_state_dict(
    smoke_initial_state
)


# Fresh optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


best_auc = -np.inf

best_epoch = 0

best_metrics = None

epochs_without_improvement = 0

history = []


print("=" * 118)
print("SIAMESE V3 k=7 — FULL RANDOM-PAIR TRAINING")
print("=" * 118)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = train_one_epoch(
        model=model,
        loader=train_pair_loader,
        dataset=train_pair_dataset,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = evaluate_pairwise(
        model=model,
        loader=val_pair_loader,
        criterion=criterion
    )


    # ========================================================
    # HISTORY
    # ========================================================

    row = {
        "epoch":
            epoch,

        "train_loss":
            train_metrics["loss"],

        "train_auc":
            train_metrics["auc"],

        "train_d_pos":
            train_metrics["d_pos"],

        "train_d_neg":
            train_metrics["d_neg"],

        "train_gap":
            train_metrics["gap"],

        "val_loss":
            val_metrics["loss"],

        "val_auc":
            val_metrics["auc"],

        "val_d_pos":
            val_metrics["d_pos"],

        "val_d_neg":
            val_metrics["d_neg"],

        "val_gap":
            val_metrics["gap"],

        "val_d_prime":
            val_metrics["d_prime"],

        "seconds":
            train_metrics["seconds"]
    }


    history.append(
        row
    )


    # ========================================================
    # CHECKPOINT
    # ========================================================

    improved = (
        val_metrics["auc"]
        >
        best_auc
        +
        MIN_DELTA
    )


    if improved:

        best_auc = (
            val_metrics["auc"]
        )

        best_epoch = epoch

        best_metrics = copy.deepcopy(
            val_metrics
        )

        epochs_without_improvement = 0


        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    copy.deepcopy(
                        model.state_dict()
                    ),

                "val_metrics":
                    best_metrics,

                "k":
                    K,

                "fcgr_size":
                    FCGR_SIZE,

                "embedding_dim":
                    EMBEDDING_DIM,

                "margin":
                    EUCLIDEAN_MARGIN,

                "learning_rate":
                    LEARNING_RATE,

                "training":
                    "random_pairs",

                "encoder":
                    "CNN_V3"
            },

            CHECKPOINT_PATH
        )


        marker = " <-- BEST"


    else:

        epochs_without_improvement += 1

        marker = ""


    print(

        f"Epoch {epoch:02d}/{MAX_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"

        f"{marker}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print()
        print(
            "Early stopping triggered."
        )

        break


# ============================================================
# SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(
    HISTORY_PATH,
    sep="\t",
    index=False
)


print()
print("=" * 76)
print("k=7 TRAINING COMPLETE")
print("=" * 76)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best Val ROC-AUC:",
    f"{best_metrics['auc']:.6f}"
)

print(
    "Best Val loss:",
    f"{best_metrics['loss']:.6f}"
)

print(
    "Best d+:",
    f"{best_metrics['d_pos']:.6f}"
)

print(
    "Best d-:",
    f"{best_metrics['d_neg']:.6f}"
)

print(
    "Best gap:",
    f"{best_metrics['gap']:.6f}"
)

print(
    "Best d-prime:",
    f"{best_metrics['d_prime']:.6f}"
)

print()
print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

SIAMESE V3 k=7 — FULL RANDOM-PAIR TRAINING
Epoch 01/40 | train loss 0.3799 | train AUC 0.6219 | val loss 0.3815 | val AUC 0.6269 | d+ 0.6164 | d- 0.7210 | gap 0.1046 | d' 0.4603 | 67.3s <-- BEST
Epoch 02/40 | train loss 0.3690 | train AUC 0.6396 | val loss 0.3778 | val AUC 0.6428 | d+ 0.6276 | d- 0.7496 | gap 0.1220 | d' 0.5171 | 68.4s <-- BEST
Epoch 03/40 | train loss 0.3664 | train AUC 0.6466 | val loss 0.3680 | val AUC 0.6438 | d+ 0.5710 | d- 0.6708 | gap 0.0998 | d' 0.5169 | 118.6s <-- BEST
Epoch 04/40 | train loss 0.3657 | train AUC 0.6477 | val loss 0.3721 | val AUC 0.6394 | d+ 0.5237 | d- 0.6168 | gap 0.0931 | d' 0.5015 | 67.8s
Epoch 05/40 | train loss 0.3619 | train AUC 0.6567 | val loss 0.3744 | val AUC 0.6325 | d+ 0.5979 | d- 0.6969 | gap 0.0990 | d' 0.4791 | 130.8s
Epoch 06/40 | train loss 0.3615 | train AUC 0.6576 | val loss 0.3671 | val AUC 0.6480 | d+ 0.5333 | d- 0.6258 | gap 0.0925 | d' 0.5336 | 67.8s <-- BEST
Epoch 07/40 | train loss 0.3593 | train AUC 0.6632 | val loss

In [20]:
# ============================================================
# LOAD BEST k=7 MODEL
# ============================================================

best_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.eval()

print("=" * 72)
print("BEST k=7 MODEL LOADED")
print("=" * 72)

print(
    "Epoch:",
    best_checkpoint["epoch"]
)

print(
    "Val ROC-AUC:",
    f"{best_checkpoint['val_metrics']['auc']:.6f}"
)

BEST k=7 MODEL LOADED
Epoch: 18
Val ROC-AUC: 0.665868


In [21]:
# ============================================================
# SINGLE FCGR DATASET
# ============================================================

class SingleFCGRDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = (
            fcgr_memmap
        )


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


    def __len__(self):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        x = np.array(
            self.fcgr_memmap[
                int(
                    self.rows[index]
                )
            ],
            dtype=np.float32,
            copy=True
        )


        return {
            "x":
                torch.from_numpy(
                    x
                ).unsqueeze(0),

            "label":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }

In [22]:
# ============================================================
# SINGLE-SAMPLE LOADERS
# ============================================================

reference_dataset = SingleFCGRDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row
)


validation_dataset = SingleFCGRDataset(
    metadata=val_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row
)


EMBEDDING_BATCH_SIZE = 128


reference_loader = DataLoader(
    reference_dataset,
    batch_size=EMBEDDING_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=EMBEDDING_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


print(
    "Reference samples:",
    len(reference_dataset)
)

print(
    "Validation samples:",
    len(validation_dataset)
)

Reference samples: 25332
Validation samples: 9753


In [23]:
# ============================================================
# EXTRACT EMBEDDINGS
# ============================================================

@torch.no_grad()
def extract_embeddings(
    encoder,
    loader
):

    encoder.eval()

    all_embeddings = []
    all_labels = []


    for batch in loader:

        x = batch["x"].to(
            DEVICE,
            non_blocking=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),
            enabled=AMP_ENABLED
        ):

            z = encoder(
                x
            )


        all_embeddings.append(
            z.float()
            .cpu()
            .numpy()
        )


        all_labels.append(
            batch["label"]
            .numpy()
        )


    return (
        np.concatenate(
            all_embeddings,
            axis=0
        ),

        np.concatenate(
            all_labels,
            axis=0
        )
    )

In [24]:
# ============================================================
# k=7 EMBEDDINGS
# ============================================================

reference_embeddings, reference_labels = (
    extract_embeddings(
        model.encoder,
        reference_loader
    )
)


val_embeddings, val_labels = (
    extract_embeddings(
        model.encoder,
        validation_loader
    )
)


print("=" * 72)
print("k=7 EMBEDDINGS")
print("=" * 72)

print(
    "Reference:",
    reference_embeddings.shape
)

print(
    "Validation:",
    val_embeddings.shape
)

print(
    "Reference mean norm:",
    np.linalg.norm(
        reference_embeddings,
        axis=1
    ).mean()
)

print(
    "Validation mean norm:",
    np.linalg.norm(
        val_embeddings,
        axis=1
    ).mean()
)

k=7 EMBEDDINGS
Reference: (25332, 128)
Validation: (9753, 128)
Reference mean norm: 1.0
Validation mean norm: 1.0


In [25]:
# ============================================================
# CLASS PROTOTYPES
# ============================================================

prototypes = []


for class_id in range(
    N_CLASSES
):

    class_embeddings = (
        reference_embeddings[
            reference_labels
            ==
            class_id
        ]
    )


    prototype = (
        class_embeddings.mean(
            axis=0
        )
    )


    prototype = (
        prototype
        /
        (
            np.linalg.norm(
                prototype
            )
            +
            1e-12
        )
    )


    prototypes.append(
        prototype
    )


prototypes = np.stack(
    prototypes,
    axis=0
)


# ============================================================
# EUCLIDEAN DISTANCE TO PROTOTYPES
# ============================================================

prototype_distances = np.linalg.norm(

    val_embeddings[
        :, None, :
    ]
    -
    prototypes[
        None, :, :
    ],

    axis=2
)


val_predictions_prototype = (
    prototype_distances.argmin(
        axis=1
    )
)


prototype_accuracy = accuracy_score(
    val_labels,
    val_predictions_prototype
)


prototype_macro_f1 = f1_score(
    val_labels,
    val_predictions_prototype,
    average="macro",
    zero_division=0
)


prototype_balanced = balanced_accuracy_score(
    val_labels,
    val_predictions_prototype
)


print("=" * 72)
print("k=7 — PROTOTYPE CLASSIFICATION")
print("=" * 72)

print(
    "Accuracy:",
    f"{prototype_accuracy:.6f}"
)

print(
    "Macro-F1:",
    f"{prototype_macro_f1:.6f}"
)

print(
    "Balanced Accuracy:",
    f"{prototype_balanced:.6f}"
)

k=7 — PROTOTYPE CLASSIFICATION
Accuracy: 0.288629
Macro-F1: 0.257295
Balanced Accuracy: 0.301769


In [26]:
# ============================================================
# REFERENCE-BANK CLASSIFICATION
# ============================================================

@torch.no_grad()
def evaluate_reference_bank(
    query_embeddings,
    query_labels,
    reference_embeddings,
    reference_labels,
    k_values=(1, 5, 10, 20, 50),
    batch_size=256
):

    k_values = sorted(
        k_values
    )

    max_k = max(
        k_values
    )


    references_by_class = {}


    for class_id in range(
        N_CLASSES
    ):

        class_refs = (
            reference_embeddings[
                reference_labels
                ==
                class_id
            ]
        )


        references_by_class[
            class_id
        ] = torch.from_numpy(
            class_refs
        ).float().to(
            DEVICE
        )


    predictions_by_k = {
        k: []
        for k in k_values
    }


    for start in range(
        0,
        len(query_embeddings),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(query_embeddings)
        )


        q = torch.from_numpy(
            query_embeddings[
                start:end
            ]
        ).float().to(
            DEVICE
        )


        nearest_by_class = []


        for class_id in range(
            N_CLASSES
        ):

            refs = (
                references_by_class[
                    class_id
                ]
            )


            # Embeddings L2-normalizzati:
            # ||q-r||² = 2 - 2(q·r)

            similarity = (
                q
                @
                refs.T
            )


            distances = torch.sqrt(
                torch.clamp(
                    2.0
                    -
                    2.0
                    *
                    similarity,
                    min=0.0
                )
                +
                1e-12
            )


            nearest = torch.topk(
                distances,
                k=max_k,
                dim=1,
                largest=False
            ).values


            nearest_by_class.append(
                nearest
            )


        # [batch, classes, max_k]

        nearest_by_class = torch.stack(
            nearest_by_class,
            dim=1
        )


        for k in k_values:

            # score classe =
            # distanza media dei k reference più vicini

            class_scores = (
                nearest_by_class[
                    :, :, :k
                ]
                .mean(
                    dim=2
                )
            )


            predictions = (
                class_scores
                .argmin(
                    dim=1
                )
            )


            predictions_by_k[k].append(
                predictions
                .cpu()
                .numpy()
            )


    results = []


    final_predictions = {}


    for k in k_values:

        preds = np.concatenate(
            predictions_by_k[k]
        )


        final_predictions[k] = preds


        results.append(
            {
                "k":
                    k,

                "accuracy":
                    accuracy_score(
                        query_labels,
                        preds
                    ),

                "macro_f1":
                    f1_score(
                        query_labels,
                        preds,
                        average="macro",
                        zero_division=0
                    ),

                "balanced_accuracy":
                    balanced_accuracy_score(
                        query_labels,
                        preds
                    )
            }
        )


    return (
        pd.DataFrame(
            results
        ),
        final_predictions
    )

In [28]:
# ============================================================
# RUN k=7 REFERENCE-BANK
# ============================================================

k7_reference_results, k7_reference_predictions = (
    evaluate_reference_bank(
        query_embeddings=val_embeddings,
        query_labels=val_labels,
        reference_embeddings=reference_embeddings,
        reference_labels=reference_labels,
        k_values=(1, 5, 10, 20, 50),
        batch_size=256
    )
)


display(
    k7_reference_results
)


best_reference_row = (
    k7_reference_results
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .iloc[0]
)


print()
print(
    "Best k:",
    int(
        best_reference_row["k"]
    )
)

print(
    "Best Accuracy:",
    f"{best_reference_row['accuracy']:.6f}"
)

print(
    "Best Macro-F1:",
    f"{best_reference_row['macro_f1']:.6f}"
)

print(
    "Best Balanced Accuracy:",
    f"{best_reference_row['balanced_accuracy']:.6f}"
)

,k,accuracy,macro_f1,balanced_accuracy
0,1,0.212447,0.209107,0.220767
1,5,0.280427,0.270492,0.287200
2,10,0.291603,0.277868,0.300214
3,20,0.303086,0.284334,0.311636
4,50,0.310264,0.284204,0.318278



Best k: 20
Best Accuracy: 0.303086
Best Macro-F1: 0.284334
Best Balanced Accuracy: 0.311636


In [29]:
# ============================================================
# FINAL k=7 SUMMARY
# ============================================================

k7_summary = {
    "fcgr_k": 7,
    "fcgr_shape": "128x128",

    "encoder": "CNN V3",
    "training": "random_pairs",
    "distance": "euclidean",
    "margin": 1.25,
    "embedding_dim": 128,

    "best_epoch": 18,

    "pairwise_val_auc": 0.665868,
    "pairwise_gap": 0.110207,
    "pairwise_d_prime": 0.601858,

    "prototype_accuracy": 0.288629,
    "prototype_macro_f1": 0.257295,
    "prototype_balanced_accuracy": 0.301769,

    "best_reference_k_macro_f1": 20,
    "reference_k20_accuracy": 0.303086,
    "reference_k20_macro_f1": 0.284334,
    "reference_k20_balanced_accuracy": 0.311636,

    "reference_k50_accuracy": 0.310264,
    "reference_k50_macro_f1": 0.284204,
    "reference_k50_balanced_accuracy": 0.318278
}


k7_summary_df = pd.DataFrame(
    [k7_summary]
)


display(
    k7_summary_df
)


summary_path = (
    ARTIFACTS_DIR
    / "k7_final_summary.tsv"
)


k7_summary_df.to_csv(
    summary_path,
    sep="\t",
    index=False
)


print(
    "Saved:",
    summary_path
)

,fcgr_k,fcgr_shape,encoder,training,distance,margin,embedding_dim,best_epoch,pairwise_val_auc,pairwise_gap,...,prototype_accuracy,prototype_macro_f1,prototype_balanced_accuracy,best_reference_k_macro_f1,reference_k20_accuracy,reference_k20_macro_f1,reference_k20_balanced_accuracy,reference_k50_accuracy,reference_k50_macro_f1,reference_k50_balanced_accuracy
0,7,128x128,CNN V3,random_pairs,euclidean,1.25,128,18,0.665868,0.110207,...,0.288629,0.257295,0.301769,20,0.303086,0.284334,0.311636,0.310264,0.284204,0.318278


Saved: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_k7_tumor_healthy\k7_final_summary.tsv
